# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

> Họ tên sinh viên: **Nguyễn Minh Quyền**
> Đọc kỹ `README.md` và `GUIDE.md` trước khi chạy. Notebook này đã hoàn thiện toàn bộ mã nguồn cho E0–E4.


## 0. Môi trường (Colab T4)
Khởi tạo GPU, cài đặt các thư viện cần thiết và thiết lập thông tin xác thực Kaggle.


In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi
!pip -q install -r requirements-colab.txt
!python -m pytest -q


In [ ]:
# Mount Google Drive để lưu adapter định kỳ phòng ngừa phiên Colab miễn phí bị ngắt
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_save_dir = Path('/content/drive/MyDrive/lab_day10_checkpoints')
    drive_save_dir.mkdir(parents=True, exist_ok=True)
    print(f'Đã mount Google Drive: {drive_save_dir}')
except Exception as e:
    drive_save_dir = Path('work/checkpoints')
    print(f'Chạy local hoặc không thể mount Drive ({e}), lưu tại: {drive_save_dir}')


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Đã nạp Kaggle credentials từ Colab Secrets.')
except Exception as e:
    print('Lưu ý: Nếu chưa cấu hình Colab Secrets, hãy chắc chắn bạn đã cấu hình ~/.kaggle/kaggle.json hoặc os.environ.')


## 1. Dữ liệu: tải và chia split theo phiên quay (session)


In [ ]:
!python scripts/prepare_data.py --kaggle


In [ ]:
# Đọc dữ liệu, kiểm tra thống kê và vẽ 6 mẫu dữ liệu có bounding box
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_entries = read_jsonl('data/splits/train.jsonl')
val_entries = read_jsonl('data/splits/val.jsonl')
test_entries = read_jsonl('data/splits/test.jsonl')
meta_info = json.loads(Path('data/splits/meta.json').read_text(encoding='utf-8'))

print(f'dataset_version: {meta_info.get("dataset_version")}')
print(f'Số lượng ảnh: Train={len(train_entries)}, Val={len(val_entries)}, Test={len(test_entries)}')

for name, entries in [('train', train_entries), ('val', val_entries), ('test', test_entries)]:
    counts = {'pothole': 0, 'crack': 0, 'manhole': 0}
    for e in entries:
        for o in e['objects']:
            counts[o['label']] = counts.get(o['label'], 0) + 1
    print(f'Split {name:5s}: {counts}')

# Trực quan hoá 6 mẫu train
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
for idx, ax in enumerate(axes.flat):
    if idx >= len(train_entries):
        break
    item = train_entries[idx]
    img = Image.open(Path('data') / item['image_path']).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for obj in item['objects']:
        xc, yc, bw, bh = obj['yolo']
        x1 = (xc - bw / 2.0) * w
        y1 = (yc - bh / 2.0) * h
        x2 = (xc + bw / 2.0) * w
        y2 = (yc + bh / 2.0) * h
        lbl = obj['label']
        c = colors.get(lbl, 'green')
        draw.rectangle([x1, y1, x2, y2], outline=c, width=3)
        draw.text((x1, max(0, y1 - 12)), lbl, fill=c)
    ax.imshow(img)
    ax.set_title(f"{item['image_id']} ({len(item['objects'])} boxes)")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi nhãn từ toạ độ chuẩn hoá YOLO `[xc, yc, w, h]` sang toạ độ nguyên `[x1, y1, x2, y2]` trên thang `0–1000`.
Hàm `objects_to_answer()` sắp xếp các box theo thứ tự trên $\rightarrow$ dưới, trái $\rightarrow$ phải (`y1, x1`).


In [ ]:
import json

def yolo_to_box(label, yolo):
    """Chuyển YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang JSON box [x1, y1, x2, y2] thang 0-1000."""
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2.0) * 1000.0))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2.0) * 1000.0))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2.0) * 1000.0))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2.0) * 1000.0))))
    return {'label': label, 'bbox_2d': [min(x1, 1000), min(y1, 1000), min(x2, 1000), min(y2, 1000)]}

def objects_to_answer(objects):
    """Sắp xếp box theo (y1, x1) và sinh chuỗi JSON rút gọn."""
    boxes = [yolo_to_box(obj['label'], obj['yolo']) for obj in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

PROMPT = (
    "Locate all road damage instances in this image: pothole, crack, and manhole. "
    "Return a JSON list of objects, each with 'label' and 'bbox_2d' on a 0-1000 scale "
    "relative to image width and height: [{\"label\": \"...\", \"bbox_2d\": [x1, y1, x2, y2]}]. "
    "Output valid JSON only, without markdown formatting or extra text."
)

# Kiểm tra hàm chuyển đổi
test_obj = [{'label': 'pothole', 'yolo': [0.5, 0.6, 0.2, 0.1]}]
ans = objects_to_answer(test_obj)
print('Prompt:', PROMPT)
print('Answer JSON:', ans)
parsed = json.loads(ans)
assert parsed[0]['bbox_2d'] == [400, 550, 600, 650]
print('Kiểm tra Bước 2: THÀNH CÔNG!')


## 3. E0: Zero-shot (Đánh giá mô hình gốc)
Nạp `Qwen3-VL-4B-Instruct` với lượng tử hoá 4-bit (NF4). Bắt buộc `llm_int8_skip_modules=['visual', 'lm_head']` để giữ ViT và Projector ở fp16.


In [ ]:
import gc
import torch
from tqdm.auto import tqdm
from transformers import BitsAndBytesConfig, AutoProcessor

try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    from transformers import AutoModelForImageTextToText as Qwen3VLForConditionalGeneration

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

print('Đang nạp mô hình E0 (zero-shot)...')
model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'

test_entries = read_jsonl('data/splits/test.jsonl')
output_e0_path = Path('work/e0.jsonl')
output_e0_path.parent.mkdir(parents=True, exist_ok=True)

e0_predictions = []
model_e0.eval()

with torch.no_grad():
    for entry in tqdm(test_entries, desc='Inference E0'):
        img = Image.open(Path('data') / entry['image_path']).convert('RGB')
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': PROMPT}
            ]
        }]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors='pt'
        )
        inputs = {k: v.to(model_e0.device) for k, v in inputs.items()}
        prompt_len = inputs['input_ids'].shape[1]
        out = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
        raw = processor.tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
        e0_predictions.append({'image_id': entry['image_id'], 'raw': raw})

with open(output_e0_path, 'w', encoding='utf-8') as f:
    for r in e0_predictions:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Đã ghi kết quả E0 vào {output_e0_path}')

# Giải phóng VRAM
del model_e0
gc.collect()
torch.cuda.empty_cache()


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


## 4. Dựng dữ liệu huấn luyện: Chat template và Mask Loss
Mỗi ảnh 640×360 tương ứng 220 token vision.
Gán `labels = -100` cho phần prompt và token ảnh để mô hình **chỉ tính loss trên câu trả lời assistant**.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, data_root='data', fraction=1.0, seed=42):
        self.data_root = Path(data_root)
        self.entries = read_jsonl(jsonl_path)
        if fraction < 1.0:
            import random
            rng = random.Random(seed)
            sample_size = max(1, int(len(self.entries) * fraction))
            self.entries = rng.sample(self.entries, sample_size)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        img = Image.open(self.data_root / entry['image_path']).convert('RGB')
        answer = objects_to_answer(entry['objects'])
        return img, PROMPT, answer, entry['image_id']

def encode_example(image, prompt, answer, proc):
    user_msg = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
    full_msg = [
        {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
    ]
    prompt_enc = proc.apply_chat_template(
        user_msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt'
    )
    full_enc = proc.apply_chat_template(
        full_msg, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors='pt'
    )
    n = prompt_enc.input_ids.shape[1]
    assert torch.equal(full_enc.input_ids[0, :n], prompt_enc.input_ids[0]), 'Prompt must be prefix of full sequence'
    labels = full_enc.input_ids.clone()
    labels[:, :n] = -100
    item = {k: v[0] for k, v in full_enc.items() if k not in ('pixel_values', 'image_grid_thw')}
    item['labels'] = labels[0]
    if 'pixel_values' in full_enc:
        item['pixel_values'] = full_enc['pixel_values']
    if 'image_grid_thw' in full_enc:
        item['image_grid_thw'] = full_enc['image_grid_thw']
    return item

def make_collate_fn(proc):
    pad_id = proc.tokenizer.pad_token_id if proc.tokenizer.pad_token_id is not None else 0
    def collate_fn(batch_items):
        encoded = [encode_example(img, p, ans, proc) for img, p, ans, _ in batch_items]
        batch = {}
        batch['input_ids'] = pad_sequence([e['input_ids'] for e in encoded], batch_first=True, padding_value=pad_id)
        batch['attention_mask'] = pad_sequence([e['attention_mask'] for e in encoded], batch_first=True, padding_value=0)
        batch['labels'] = pad_sequence([e['labels'] for e in encoded], batch_first=True, padding_value=-100)
        if 'mm_token_type_ids' in encoded[0]:
            batch['mm_token_type_ids'] = pad_sequence([e['mm_token_type_ids'] for e in encoded], batch_first=True, padding_value=0)
        if 'pixel_values' in encoded[0]:
            batch['pixel_values'] = torch.cat([e['pixel_values'] for e in encoded], dim=0)
        if 'image_grid_thw' in encoded[0]:
            batch['image_grid_thw'] = torch.cat([e['image_grid_thw'] for e in encoded], dim=0)
        return batch
    return collate_fn

# Kiểm tra mã hoá 1 mẫu
demo_ds = RoadDamageDataset('data/splits/train.jsonl')
sample_img, sample_p, sample_ans, _ = demo_ds[0]
sample_encoded = encode_example(sample_img, sample_p, sample_ans, processor)
ans_tokens = [tok.item() for tok in sample_encoded['labels'] if tok.item() != -100]
decoded_label = processor.tokenizer.decode(ans_tokens, skip_special_tokens=False)
print('Số token nhãn học (labels != -100):', len(ans_tokens))
print('Nội dung giải mã nhãn:', decoded_label)
assert sample_ans in decoded_label
print('Kiểm tra Bước 4: THÀNH CÔNG!')


## 5. Setup LoRA / Đóng băng cho E1, E2, E3
- **E1**: LoRA chỉ LLM (252 modules target).
- **E2**: E1 + mở đầy đủ projector (`visual.merger`, `visual.deepstack_merger_list`, ~109M tham số float32).
- **E3**: E2 + LoRA trên các khối attention và MLP của ViT (96 modules, tổng 348 modules target).


In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()

    llm_targets = r'.*language_model.*(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)'
    vit_targets = r'.*visual.*blocks.*(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))'

    if setup in ('E1', 'E2'):
        target_modules = llm_targets
    elif setup == 'E3':
        target_modules = f'({llm_targets}|{vit_targets})'
    else:
        raise ValueError(f'Setup không hợp lệ: {setup}')

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_modules,
        bias='none',
        task_type='CAUSAL_LM'
    )
    peft_model = get_peft_model(model, lora_cfg)

    if setup in ('E2', 'E3'):
        proj_count = 0
        for name, param in peft_model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.requires_grad = True
                proj_count += param.numel()
        print(f'[{setup}] Đã mở Projector: {proj_count:,} tham số')

    # Chuyển TẤT CẢ tham số trainable (LoRA + Projector) sang float32 để GradScaler unscale an toàn không lỗi FP16
    for param in peft_model.parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f'[{setup}] Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)')
    return peft_model


## 6. Vòng huấn luyện tối ưu
- AdamW (`lr=2e-4`, `weight_decay=0.0`), Cosine schedule với 5% warmup (15 bước).
- 300 bước tối ưu, tích luỹ gradient (batch=2, accum=4 $\rightarrow$ batch hiệu dụng=8).
- Lưu LoRA adapter và lưu riêng file `projector_weights.pt` cho E2/E3.


In [ ]:
import time
import shutil
from transformers import get_cosine_schedule_with_warmup

def save_model_checkpoint(peft_model, save_dir, setup):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    peft_model.save_pretrained(save_path)
    if setup in ('E2', 'E3'):
        extra = {
            k: v.cpu() for k, v in peft_model.named_parameters()
            if ('visual.merger' in k or 'visual.deepstack_merger_list' in k) and v.requires_grad
        }
        torch.save(extra, save_path / 'projector_weights.pt')
        print(f'Đã lưu projector_weights.pt vào {save_path}')

def train_experiment(setup='E1', max_steps=300, batch_size=2, grad_accum=4, lr=2e-4, r=16, alpha=32, fraction=1.0):
    print(f'\n========== BẮT ĐẦU HUẤN LUYỆN {setup} ==========')
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    peft_model = apply_setup(model, setup, r=r, alpha=alpha)
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    peft_model.config.use_cache = False

    for p in peft_model.parameters():
        if p.requires_grad:
            p.data = p.data.to(torch.float32)

    trainable_params_count = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]

    ds = RoadDamageDataset('data/splits/train.jsonl', fraction=fraction)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=make_collate_fn(processor))

    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.0)
    warmup_steps = int(max_steps * 0.05)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max_steps)
    scaler = torch.amp.GradScaler('cuda')

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    peft_model.train()

    step = 0
    accum_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(total=max_steps, desc=f'Train {setup}')
    loader_iter = iter(loader)

    while step < max_steps:
        for _ in range(grad_accum):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(loader)
                batch = next(loader_iter)

            batch = {k: v.to(peft_model.device) for k, v in batch.items()}
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = peft_model(**batch)
                loss = out.loss / grad_accum

            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss {loss.item()} không hữu hạn! Dừng train.')
            scaler.scale(loss).backward()
            accum_loss += loss.item() * grad_accum

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        step += 1
        pbar.set_postfix({'loss': f'{accum_loss / grad_accum:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
        accum_loss = 0.0
        pbar.update(1)

        if step % 100 == 0:
            save_model_checkpoint(peft_model, f'work/checkpoints/{setup}_step{step}', setup)

    pbar.close()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    final_dir = Path(f'work/checkpoints/{setup}_final')
    save_model_checkpoint(peft_model, final_dir, setup)

    if drive_save_dir.exists():
        shutil.copytree(final_dir, drive_save_dir / f'{setup}_final', dirs_exist_ok=True)
        print(f'Đã sao lưu {setup} sang Google Drive')

    meta = {
        'trainable_params': trainable_params_count,
        'peak_vram_gb': round(peak_vram_gb, 3),
        'train_minutes': round(train_minutes, 2),
        'adapter_dir': str(final_dir)
    }
    del peft_model, model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Kết thúc {setup}: params={meta["trainable_params"]:,}, VRAM={meta["peak_vram_gb"]}GB, time={meta["train_minutes"]}min\n')
    return meta


## 7. Suy luận và Chấm điểm E1–E3
Nạp model + adapter + projector weights (nếu có). Suy luận trên test $\rightarrow$ `work/eN.jsonl` $\rightarrow$ chấm điểm bằng `score.py` với metadata.


In [ ]:
from peft import PeftModel

def evaluate_experiment(setup, adapter_dir, meta, out_name=None):
    name = out_name or setup
    print(f'\n--- Đang đánh giá {name} trên tập test ---')
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    eval_model = PeftModel.from_pretrained(base_model, adapter_dir)
    proj_path = Path(adapter_dir) / 'projector_weights.pt'
    if proj_path.exists():
        extra = torch.load(proj_path, map_location='cpu')
        eval_model.load_state_dict(extra, strict=False)
        print(f'Đã nạp lại projector weights từ {proj_path}')

    eval_model.eval()
    preds = []
    test_entries = read_jsonl('data/splits/test.jsonl')
    pred_file = Path(f'work/{name.lower()}.jsonl')
    pred_file.parent.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for entry in tqdm(test_entries, desc=f'Eval {name}'):
            img = Image.open(Path('data') / entry['image_path']).convert('RGB')
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img},
                    {'type': 'text', 'text': PROMPT}
                ]
            }]
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors='pt'
            )
            inputs = {k: v.to(eval_model.device) for k, v in inputs.items()}
            prompt_len = inputs['input_ids'].shape[1]
            out = eval_model.generate(**inputs, max_new_tokens=256, do_sample=False)
            raw = processor.tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
            preds.append({'image_id': entry['image_id'], 'raw': raw})

    with open(pred_file, 'w', encoding='utf-8') as f:
        for r in preds:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    del eval_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    evidence_file = Path(f'report/evidence/{name.lower()}.json')
    evidence_file.parent.mkdir(parents=True, exist_ok=True)
    cmd = (
        f'python scripts/score.py --pred {pred_file} --name {name} --out {evidence_file} '
        f'--meta trainable_params={meta["trainable_params"]} '
        f'--meta peak_vram_gb={meta["peak_vram_gb"]} '
        f'--meta train_minutes={meta["train_minutes"]}'
    )
    print('Lệnh score:', cmd)
    os.system(cmd)


In [ ]:
# Huấn luyện và đánh giá E1 (LoRA LLM)
meta_e1 = train_experiment(setup='E1')
evaluate_experiment('E1', meta_e1['adapter_dir'], meta_e1)


In [ ]:
# Huấn luyện và đánh giá E2 (E1 + Projector)
meta_e2 = train_experiment(setup='E2')
evaluate_experiment('E2', meta_e2['adapter_dir'], meta_e2)


In [ ]:
# Huấn luyện và đánh giá E3 (E2 + LoRA ViT)
meta_e3 = train_experiment(setup='E3')
evaluate_experiment('E3', meta_e3['adapter_dir'], meta_e3)


## 8. E4: Ablation Study & Trực quan hoá kết quả
Ablation so sánh **LoRA Rank** ($r=4$ vs $r=64$, $\alpha = 2r$) trên cấu hình E1.


In [ ]:
# 8.1. E4 Rank=4
meta_e4_r4 = train_experiment(setup='E1', r=4, alpha=8)
evaluate_experiment('E1', meta_e4_r4['adapter_dir'], meta_e4_r4, out_name='e4_rank_4')


In [ ]:
# 8.2. E4 Rank=64
meta_e4_r64 = train_experiment(setup='E1', r=64, alpha=128)
evaluate_experiment('E1', meta_e4_r64['adapter_dir'], meta_e4_r64, out_name='e4_rank_64')


In [ ]:
# 8.3. Trực quan hoá ≥ 6 ảnh có bounding box dự đoán vs ground-truth (lưu vào report/images/)
from scripts.score import parse_output

viz_dir = Path('report/images')
viz_dir.mkdir(parents=True, exist_ok=True)

test_entries = read_jsonl('data/splits/test.jsonl')
e1_preds = {r['image_id']: r['raw'] for r in read_jsonl('work/e1.jsonl')}

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
for idx, ax in enumerate(axes.flat):
    if idx >= len(test_entries):
        break
    entry = test_entries[idx]
    img = Image.open(Path('data') / entry['image_path']).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = img.size
    # Vẽ Ground Truth (Xanh lá)
    for obj in entry['objects']:
        xc, yc, bw, bh = obj['yolo']
        x1, y1 = (xc - bw / 2.0) * w, (yc - bh / 2.0) * h
        x2, y2 = (xc + bw / 2.0) * w, (yc + bh / 2.0) * h
        draw.rectangle([x1, y1, x2, y2], outline='lime', width=3)
        draw.text((x1, max(0, y1 - 12)), f"GT: {obj['label']}", fill='lime')
    # Vẽ Prediction (Đỏ)
    raw = e1_preds.get(entry['image_id'], '')
    parsed = parse_output(raw)
    if parsed:
        for p in parsed:
            px1, py1, px2, py2 = p['bbox']
            rx1, ry1 = px1 * w / 1000.0, py1 * h / 1000.0
            rx2, ry2 = px2 * w / 1000.0, py2 * h / 1000.0
            draw.rectangle([rx1, ry1, rx2, ry2], outline='red', width=2)
            draw.text((rx1, min(h - 12, ry2)), f"Pred: {p['label']}", fill='red')
    ax.imshow(img)
    ax.set_title(f"{entry['image_id']} (Xanh: GT, Đỏ: E1)")
    ax.axis('off')
    img.save(viz_dir / f'sample_{idx+1}_{entry["image_id"]}.jpg')

plt.tight_layout()
plt.savefig(viz_dir / 'visualization_grid.jpg', dpi=150)
plt.show()
print(f'Đã lưu 6 ảnh trực quan hoá vào {viz_dir}')


## 9. Kiểm tra bài nộp (`check_submission.py`)


In [ ]:
!python scripts/check_submission.py
